# VLM 학습 데이터 준비

실제 도메인 데이터를 구하는 대신, `PIL`로 즉석에서 만드는 장난감(toy) **"제품 불량 검사"** 데이터셋으로 fine-tuning 데이터 포맷을 익힙니다.

- 이미지: 사각형 "제품" + (있다면) 모서리의 빨간 스크래치
- 라벨: 불량 유무, 불량이면 어느 모서리인지
- 포맷: `image` + `conversations` instruction-following JSON (LLM instruction tuning 포맷에 이미지가 추가된 형태)

## 사전 준비
```bash
uv sync
```


In [ ]:
import vlm_finetune_utils as vu


## 1. 데이터셋 생성
train 40장 / val 10장, 각각 절반은 정상 · 절반은 불량(모서리 랜덤)으로 생성합니다.

In [ ]:
vu.make_toy_defect_dataset("data", {"train": 40, "val": 10}, seed=0)
print("생성 완료")


## 2. 샘플 확인

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

records = vu.load_jsonl("data/train/annotations.jsonl")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, r in zip(axes, records[:4]):
    img = Image.open(f"data/train/{r['image']}")
    ax.imshow(img)
    ax.set_title(r["conversations"][1]["value"], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
records[0]


## 3. 데이터 품질 체크리스트
fine-tuning 데이터에서 중요한 **다양성 / 일관성 / 클래스 불균형**을 코드로 직접 확인합니다.

In [ ]:
from collections import Counter

defect_counts = Counter(r["label"]["has_defect"] for r in records)
corner_counts = Counter(r["label"]["corner"] for r in records if r["label"]["has_defect"])

print("정상/불량 비율:", defect_counts)
print("모서리별 분포:", corner_counts)


정상:불량 비율이 1:1에 가깝고, 불량 모서리도 특정 방향에 쏠리지 않아야 합니다(그래야 모델이 '무조건 우측 상단'처럼 위치를 외워버리는 것을 방지). 치우쳐 있다면 `seed`나 개수를 조정해 다시 생성하세요.

## 정리
- `data/train/annotations.jsonl`, `data/val/annotations.jsonl`에 학습/검증 데이터가 저장됨
- 다음 노트북(`2. QLoRA Fine-tuning.ipynb`)에서 이 데이터로 실제 학습을 진행